Setup:

In [19]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
import statsmodels.api as sm

Load the efficiency and decentralization dataframes using their respective methods from efficiency.py and decentralization.py

In [20]:
from src.signals.efficiency import load_metrics as load_efficiency
from src.signals.decentralization import load_metrics as load_decentralization

eff_df = load_efficiency()
dec_df = load_decentralization()

print("Efficiency dataset:")
print(eff_df[['event_name', 'convergence_speed', 'kalshi_spread', 'nakamoto_pre', 'gini_pre', 'hhi_pre']].to_string())
print(f"\nDecentralization dataset:")
print(dec_df.describe())

Efficiency dataset:
                             event_name  convergence_speed  kalshi_spread  nakamoto_pre  gini_pre     hhi_pre
0                       trump_wins_2024           0.002600       0.094000      7.000000  0.802538  565.212240
1              harris_wins_popular_vote           0.001834       0.134429      7.000000  0.802538  565.212240
2                  fed_holds_march_2025           0.001000       0.002571      6.000000  0.826547  637.772415
3           chat_gpt_best_ai_march_2025           0.009161       0.024500      6.000000  0.823404  617.906680
4  liberal_majority_next_govt_of_canada           0.026619       0.017857      6.000000  0.818958  582.397992
5              fed_holds_september_2025           0.001000       0.011286      7.000000  0.820689  563.146206
6               fed_holds_december_2025           0.001000       0.009071      7.000000  0.820994  564.565874
7     bad_bunny_top_spotify_artist_2025           0.002200       0.008143      7.000000  0.821851  5

Plot the Nakomoto coefficient time series to visualize how it changed over the study period (January 2024 to February 2026). On the plot, mark where each event falls.

In [21]:
fig = go.Figure()

# nakamoto time series
fig.add_trace(go.Scatter(
  x=dec_df['date'],
  y=dec_df['nakamoto'],
  mode='lines',
  name='Nakamoto Coefficient',
  line=dict(color='royalblue', width=1.5)
))

# mark each event with a colored dot on the line
colors = px.colors.qualitative.Set2

for i, row in eff_df.iterrows():
  event_date = pd.Timestamp(row['resolution_time']).tz_localize(None)
  
  # find the nakamoto value on or nearest to the event date
  nearest = dec_df.iloc[(dec_df['date'] - event_date).abs().argsort()[:1]]
  nakamoto_at_event = nearest['nakamoto'].values[0]
    
  fig.add_trace(go.Scatter(
    x=[event_date],
    y=[nakamoto_at_event],
    mode='markers',
    name=row['event_name'].replace('_', ' '),
    marker=dict(size=10, color=colors[i % len(colors)], symbol='circle'),
  ))

fig.update_layout(
  title='Polygon Validator Nakamoto Coefficient: Jan 2024 to Feb 2026',
  xaxis_title='Date',
  yaxis_title='Nakamoto Coefficient',
  yaxis=dict(range=[4, 10], dtick=1),
  hovermode='x unified',
  height=500,
  legend=dict(
    orientation='v',
    x=1.02,
    y=1,
    font=dict(size=10)
  )
)

fig.show()

Perform both Pearson and Spearman correlation analyses between the average nakamoto coefficient prior to resolution (nakamoto_pre) and both the convergence speed and the kalshi spread.

In [22]:
# pearson correlation
pearson_cs, p_pearson_cs = stats.pearsonr(eff_df['nakamoto_pre'], eff_df['convergence_speed'])
pearson_ks, p_pearson_ks = stats.pearsonr(eff_df['nakamoto_pre'], eff_df['kalshi_spread'])

# spearman correlation
spearman_cs, p_spearman_cs = stats.spearmanr(eff_df['nakamoto_pre'], eff_df['convergence_speed'])
spearman_ks, p_spearman_ks = stats.spearmanr(eff_df['nakamoto_pre'], eff_df['kalshi_spread'])

print("Nakamoto vs Convergence Speed:")
print(f"Pearson: r = {pearson_cs:.4f}, p = {p_pearson_cs:.4f}")
print(f"Spearman: r = {spearman_cs:.4f}, p = {p_spearman_cs:.4f}")

print("\nNakamoto vs Kalshi Spread:")
print(f"Pearson: r = {pearson_ks:.4f}, p = {p_pearson_ks:.4f}")
print(f"Spearman: r = {spearman_ks:.4f}, p = {p_spearman_ks:.4f}")

Nakamoto vs Convergence Speed:
Pearson: r = -0.5462, p = 0.1024
Spearman: r = -0.3653, p = 0.2993

Nakamoto vs Kalshi Spread:
Pearson: r = 0.1261, p = 0.7284
Spearman: r = 0.2032, p = 0.5733


Plot nakamoto_pre against convergence_speed and nakamoto_pre against kalshi_spread to visualize the regression lines and the strength of the variables' correlation.

In [ ]:
def scatter_with_regression(x, y, labels, x_title, y_title, title):
  fig = go.Figure()

  # scatter points
  fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='markers+text',
    text=labels,
    textposition='top right',
    textfont=dict(size=9),
    marker=dict(size=10, color='royalblue'),
    name='Events'
  ))

  # regression line
  m, b = np.polyfit(x, y, 1)
  x_line = np.linspace(min(x), max(x), 100)
  y_line = m * x_line + b

  fig.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode='lines',
    line=dict(color='firebrick', dash='dash', width=1.5),
    name=f'OLS fit (slope={m:.4f})'
  ))

  fig.update_layout(
    title=title,
    xaxis_title=x_title,
    yaxis_title=y_title,
    hovermode='closest',
    height=500
  )
  return fig

labels = [
  'Trump 2024', 'Harris Popular Vote', 'Fed Mar 25', 'ChatGPT Best AI',
  'Canada Liberal Majority', 'Fed Sep 25', 'Fed Dec 25', 'Bad Bunny',
  'Recession', 'Govt Shutdown'
]

# plot 1 — nakamoto vs convergence speed
fig1 = scatter_with_regression(
  x=eff_df['nakamoto_pre'].tolist(),
  y=eff_df['convergence_speed'].tolist(),
  labels=labels,
  x_title='Nakamoto Coefficient (7-day pre-resolution average)',
  y_title='Convergence Speed (mean abs deviation from resolved value)',
  title='Nakamoto Coefficient vs Polymarket Convergence Speed'
)
fig1.show()

# plot 2 — nakamoto vs kalshi spread
fig2 = scatter_with_regression(
  x=eff_df['nakamoto_pre'].tolist(),
  y=eff_df['kalshi_spread'].tolist(),
  labels=labels,
  x_title='Nakamoto Coefficient (7-day pre-resolution average)',
  y_title='Kalshi Spread (mean abs deviation between platforms)',
  title='Nakamoto Coefficient vs Polymarket-Kalshi Spread',
)
fig2.show()

Observations:
- Convergence speed regression slope is 0.0003, essentially flat, with three events clustered at the floor (0.001) regardless of Nakamoto value, visually confirming the null result
- Kalshi spread chart shows the two election markets (Trump and Harris) as clear outliers at ~0.094-0.098, with all other events clustered near 0
- The positive regression slope of 0.0039 on the Kalshi spread chart is driven by the election outliers, not a genuine trend
- Fed Mar 25 at Nakamoto=6 has the lowest spread, consistent with the hypothesis, but a single data point is insufficient to draw conclusions
- The election markets are the key confound: their large spreads likely reflect unique market characteristics (massive volume, international vs US participant base, high political uncertainty) rather than validator concentration

OLS regression:

In [17]:
for metric, label in [('convergence_speed', 'Convergence Speed'), ('kalshi_spread', 'Kalshi Spread')]:
  X = sm.add_constant(eff_df['nakamoto_pre'])
  y = eff_df[metric]
  model = sm.OLS(y, X).fit()
  print(f"\nOLS: Nakamoto vs {label}")
  print(f"coef: {model.params['nakamoto_pre']:.6f}")
  print(f"p-value: {model.pvalues['nakamoto_pre']:.4f}")
  print(f"r-squared: {model.rsquared:.4f}")


OLS: Nakamoto vs Convergence Speed
coef: 0.000262
p-value: 0.5677
r-squared: 0.0574

OLS: Nakamoto vs Kalshi Spread
coef: 0.003879
p-value: 0.8879
r-squared: 0.0036


Neither regression produces a statistically significant result. The R-squared values (0.057 and 0.004) indicate that the Nakamoto coefficient explains essentially none of the variation in either efficiency metric. Additionally, the p-values (0.57 and 0.89) are far above any conventional significance threshold.

With n = 10 observations and an independent variable ranging only from 6 to 8, detecting a small effect would require a much larger sample. These results are consistent with a null effect, but cannot rule out a real relationship that this study lacked the power to detect.

Now plot the convergence windows (7 days pre-resolution + 48 hours post) to visualize the Polymarket-Kalshi spread in the pre-resolution timeframe.

In [26]:
from src.collect.polymarket import get_price_history as pm_get
from src.collect.kalshi import get_price_history as kal_get
from plotly.subplots import make_subplots

events = pd.read_csv('../data/events.csv')

fig = make_subplots(
  rows=5, cols=2,
  subplot_titles=[e.replace('_', ' ') for e in events['event_name'].tolist()],
  vertical_spacing=0.1,
  horizontal_spacing=0.08
)

for i, row in events.iterrows():
  r = (i // 2) + 1
  c = (i % 2) + 1
  
  event_name = row['event_name']
  resolved_value = int(row['resolved_value'])
  resolution_time = pd.Timestamp(int(float(row['resolution_ts'])), unit='s', tz='UTC')
  window_start = resolution_time - pd.Timedelta(days=7)
  window_end = resolution_time + pd.Timedelta(hours=48)

  pm_df = pm_get(event_name)
  kal_df = kal_get(event_name)

  pm_window = pm_df[(pm_df['timestamp'] >= window_start) & (pm_df['timestamp'] <= window_end)]
  kal_window = kal_df[(kal_df['timestamp'] >= window_start) & (kal_df['timestamp'] <= window_end)]

  # polymarket line
  fig.add_trace(go.Scatter(
    x=pm_window['timestamp'],
    y=pm_window['price'],
    mode='lines',
    name='Polymarket',
    line=dict(color='royalblue', width=1.5),
    showlegend=(i == 0)
  ), row=r, col=c)

  # kalshi line
  fig.add_trace(go.Scatter(
    x=kal_window['timestamp'],
    y=kal_window['price'],
    mode='lines+markers',
    name='Kalshi',
    line=dict(color='firebrick', width=1.5),
    marker=dict(size=4),
    showlegend=(i == 0)
  ), row=r, col=c)

  # resolved value line
  fig.add_hline(
    y=resolved_value,
    line_dash='dash',
    line_color='green',
    line_width=1,
    row=r, col=c
  )

  # resolution timestamp
  fig.add_vline(
    x=resolution_time.timestamp() * 1000,
    line_dash='dot',
    line_color='grey',
    line_width=1,
    row=r, col=c
  )

fig.update_yaxes(range=[-0.05, 1.05])
fig.update_layout(
  title='Convergence Windows: All 10 Events (7 days pre + 48hr post resolution)',
  height=1750,
  hovermode='x unified'
)

fig.show()

### What the results show
- No statistically significant relationship found between Nakamoto coefficient and either efficiency metric
- Consistent with two explanations: the effect genuinely does not exist at this scale, or the study is underpowered to detect it

### Key confounds
- Event type dominates the Kalshi spread signal; election markets have spreads 10x larger than all other events, likely due to market characteristics rather than validator concentration
- Nakamoto coefficient ranged only from 6 to 8 across 790 days, providing insufficient variation to detect a relationship
- n = 10 means a single outlier still has disproportionate influence on all regression results

### What a better-powered study would look like
- Cross-chain comparison where validator concentration varies more dramatically
- Larger event sample (50+) with controls for event type and volume
- Top validator stake share as an alternative concentration measure, since MEV does not require majority control